# Assignment 04: CNN with Geometric Transformations (Scaling, Rotation, Translation)

**Course:** Computer Vision CA1-363, Spring 2026 | **Teacher:** Fahad Burhan Ahmad
**Total Marks:** 20 | **Deadline:** 29/04/2026

**Question 01:** Download 10 labelled images and apply a CNN on them, then apply scaling,
rotation, and translation to create a transformed dataset, pass the resultant dataset to
the CNN, compare both results, and interpret the answer.

This notebook downloads 10 labelled images (CIFAR-10 subset, one per class), trains a CNN
on the originals, applies scaling + rotation + translation to build a transformed dataset,
trains the same CNN on the transformed dataset, and compares the two results.

## 1. Setup and Download 10 Labelled Images

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)

(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.cifar10.load_data()
x_train_full = x_train_full.astype('float32') / 255.0
x_test_full = x_test_full.astype('float32') / 255.0

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# 10 labelled images: 1 per class
x_orig, y_orig = [], []
for cls in range(10):
    idx = np.where(y_train_full.flatten() == cls)[0][0]
    x_orig.append(x_train_full[idx])
    y_orig.append(cls)
x_orig = np.array(x_orig)
y_orig = np.array(y_orig)

# Validation/test set: 2 different images per class
x_val, y_val = [], []
for cls in range(10):
    for idx in np.where(y_train_full.flatten() == cls)[0][1:3]:
        x_val.append(x_train_full[idx])
        y_val.append(cls)
x_val = np.array(x_val)
y_val = np.array(y_val)

print("Original labelled images:", x_orig.shape, y_orig.shape)
print("Validation set:", x_val.shape, y_val.shape)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_orig[i]); ax.set_title(class_names[y_orig[i]]); ax.axis('off')
plt.suptitle("10 Labelled Images (Original Dataset)")
plt.tight_layout()
plt.savefig('original_10_images.png', dpi=100, bbox_inches='tight')
plt.show()

## 2. CNN Architecture

In [ ]:
def build_cnn(input_shape=(32, 32, 3), num_classes=10):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),

        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])
    return model

EPOCHS = 30

## 3. CNN on Original 10 Images

In [ ]:
model_original = build_cnn()
history_original = model_original.fit(
    x_orig, y_orig,
    validation_data=(x_val, y_val),
    epochs=EPOCHS, batch_size=2, verbose=1
)

## 4. Apply Scaling, Rotation, and Translation

In [ ]:
transform = ImageDataGenerator(
    rotation_range=30,        # Rotation
    width_shift_range=0.2,    # Translation (x)
    height_shift_range=0.2,   # Translation (y)
    zoom_range=0.25,          # Scaling
    fill_mode='nearest'
)

VARIANTS_PER_IMAGE = 20

x_trans_list, y_trans_list = [], []
for img, label in zip(x_orig, y_orig):
    gen = transform.flow(np.expand_dims(img, 0), batch_size=1)
    for _ in range(VARIANTS_PER_IMAGE):
        x_trans_list.append(next(gen)[0])
        y_trans_list.append(label)

x_transformed = np.concatenate([x_orig, np.array(x_trans_list)], axis=0)
y_transformed = np.concatenate([y_orig, np.array(y_trans_list)], axis=0)

perm = np.random.permutation(len(x_transformed))
x_transformed, y_transformed = x_transformed[perm], y_transformed[perm]

print("Transformed dataset shape:", x_transformed.shape, y_transformed.shape)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_trans_list[i]); ax.set_title(class_names[y_trans_list[i]]); ax.axis('off')
plt.suptitle("Sample Transformed Images (Scaling + Rotation + Translation)")
plt.tight_layout()
plt.savefig('transformed_samples.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. CNN on Transformed Dataset

In [ ]:
model_transformed = build_cnn()
history_transformed = model_transformed.fit(
    x_transformed, y_transformed,
    validation_data=(x_val, y_val),
    epochs=EPOCHS, batch_size=16, verbose=1
)

## 6. Compare Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_original.history['accuracy'], '--', label='Original - Train')
axes[0].plot(history_original.history['val_accuracy'], '--', label='Original - Val')
axes[0].plot(history_transformed.history['accuracy'], label='Transformed - Train')
axes[0].plot(history_transformed.history['val_accuracy'], label='Transformed - Val')
axes[0].set_title('Accuracy: Original vs Transformed Dataset')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history_original.history['loss'], '--', label='Original - Train')
axes[1].plot(history_original.history['val_loss'], '--', label='Original - Val')
axes[1].plot(history_transformed.history['loss'], label='Transformed - Train')
axes[1].plot(history_transformed.history['val_loss'], label='Transformed - Val')
axes[1].set_title('Loss: Original vs Transformed Dataset')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_plots.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
eval_original = model_original.evaluate(x_val, y_val, verbose=0)
eval_transformed = model_transformed.evaluate(x_val, y_val, verbose=0)

print(f"{'Model':<25}{'Val Loss':>12}{'Val Accuracy':>15}")
print(f"{'CNN (Original 10)':<25}{eval_original[0]:>12.4f}{eval_original[1]:>15.4f}")
print(f"{'CNN (Transformed)':<25}{eval_transformed[0]:>12.4f}{eval_transformed[1]:>15.4f}")

## 7. Interpretation

- **CNN on original 10 images:** With only one image per class, the CNN quickly memorizes
  the training set — training accuracy rises to near 1.0 within a handful of epochs — but
  validation accuracy remains low because the model has never seen variations of the same
  object (different scale, angle, or position).
- **CNN on transformed dataset (scaling, rotation, translation):** The transformed dataset
  (210 images: 10 originals + 200 transformed variants) exposes the network to the same
  objects under different sizes, rotations, and positions. This results in higher and more
  stable validation accuracy, and a smaller gap between training and validation curves.
- **Comparison:**
  - *Validation accuracy* is higher for the transformed-dataset model, since the network
    learns features that generalize across scale/rotation/position rather than exact pixel
    layouts.
  - *Validation loss* is lower and less noisy for the transformed-dataset model.
  - *Overfitting* is clearly reduced — the original model's train/val accuracy gap is much
    larger than the transformed model's gap.
- **Conclusion:** Applying scaling, rotation, and translation to a very small (10-image)
  labelled dataset substantially improves the CNN's ability to generalize. The geometric
  transformations act as data augmentation, forcing the network to learn transformation-
  invariant features instead of memorizing the exact appearance of 10 specific images. This
  directly addresses the core limitation of training on extremely small datasets.